#### RAG INGESTION PIPELINE: DATA TO VECTOR

In [1]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from pathlib import Path

/home/anonymous/projects/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("HF_TOKEN"):
    print("HF_TOKEN not set. Add it to your .env file.")

In [3]:
# Read all the pdf inside the directory
def process_pdfs(pdf_directory: str):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} Pdf files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
all_pdf_documents = process_pdfs("../data")

Found 3 Pdf files to process

Processing: EventX-finalppt.pdf
Loaded 26 pages

Processing: EventX_OOAD (1).pdf
Loaded 18 pages

Processing: Final_copy-5.pdf
Loaded 55 pages

Total documents loaded: 99


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'EventX-ppt-proposal', 'source': '../data/pdf/EventX-finalppt.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'EventX-finalppt.pdf', 'file_type': 'pdf'}, page_content='Minor Project on\n: One Place For All Events\nTRIBHUV AN UNIVERSITY\nINSTITUTE OF ENGINEERING\nAashish Adhikari\nAshim Shrestha \nSagun Paudel \nSaugat Adhikari\n[PAS079BCT003]\n[PAS079BCT007]\n[PAS079BCT032]\n[PAS079BCT036]\nDepartment Of Electronics And Computer Engineering\n13 May, 2026\n Presented By: \n Supervised By: Asst. Prof. Nabin Lamichhane'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'EventX-ppt-proposal', 'source': '../data/pdf/EventX-finalppt.pdf', 'total_pages': 26, 'page': 1, 'page_label': '2', 'source_file': 'EventX-finalppt.pdf', 'file_type': 'pdf'}, page_content='Institute of Engineering, Tribhuvan University, Nepal\nProblem Statement\n•   Eve

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """ Split Documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks[:5]

Split 99 documents into 139 chunks

Example Chunk:
Content: Minor Project on
: One Place For All Events
TRIBHUV AN UNIVERSITY
INSTITUTE OF ENGINEERING
Aashish Adhikari
Ashim Shrestha 
Sagun Paudel 
Saugat Adhikari
[PAS079BCT003]
[PAS079BCT007]
[PAS079BCT032]
[...
Metadata: {'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'EventX-ppt-proposal', 'source': '../data/pdf/EventX-finalppt.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'EventX-finalppt.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'EventX-ppt-proposal', 'source': '../data/pdf/EventX-finalppt.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'EventX-finalppt.pdf', 'file_type': 'pdf'}, page_content='Minor Project on\n: One Place For All Events\nTRIBHUV AN UNIVERSITY\nINSTITUTE OF ENGINEERING\nAashish Adhikari\nAshim Shrestha \nSagun Paudel \nSaugat Adhikari\n[PAS079BCT003]\n[PAS079BCT007]\n[PAS079BCT032]\n[PAS079BCT036]\nDepartment Of Electronics And Computer Engineering\n13 May, 2026\n Presented By: \n Supervised By: Asst. Prof. Nabin Lamichhane'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'EventX-ppt-proposal', 'source': '../data/pdf/EventX-finalppt.pdf', 'total_pages': 26, 'page': 1, 'page_label': '2', 'source_file': 'EventX-finalppt.pdf', 'file_type': 'pdf'}, page_content='Institute of Engineering, Tribhuvan University, Nepal\nProblem Statement\n•   Eve

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handling Document Embedding Generation using setence transformers"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding Manager

        Args:
            model_name: Hugging Face Model name for sentence embeddings."
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading Model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for list of texts

        Args:
            texts (List[str]): List of text strings to embed

        Returns:
            np.ndarray: numpy array with shape(len(texts), embedding_dim )
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [9]:
## initialize the embedding manager
embedding_manager= EmbeddingManager()
embedding_manager

Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11424.39it/s]


Model loaded successfully. Embedding dimension: 384


#### VectorStore

In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent chormaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF docuement embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error intializing vector store: {e}")
            raise
        
    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store
        n: 0
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []        
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Convert numpy array Embeddings to python list
            embeddings_list.append(embedding.tolist())
            
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore
        

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 139


In [11]:
chunks[:5]

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'EventX-ppt-proposal', 'source': '../data/pdf/EventX-finalppt.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'EventX-finalppt.pdf', 'file_type': 'pdf'}, page_content='Minor Project on\n: One Place For All Events\nTRIBHUV AN UNIVERSITY\nINSTITUTE OF ENGINEERING\nAashish Adhikari\nAshim Shrestha \nSagun Paudel \nSaugat Adhikari\n[PAS079BCT003]\n[PAS079BCT007]\n[PAS079BCT032]\n[PAS079BCT036]\nDepartment Of Electronics And Computer Engineering\n13 May, 2026\n Presented By: \n Supervised By: Asst. Prof. Nabin Lamichhane'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'EventX-ppt-proposal', 'source': '../data/pdf/EventX-finalppt.pdf', 'total_pages': 26, 'page': 1, 'page_label': '2', 'source_file': 'EventX-finalppt.pdf', 'file_type': 'pdf'}, page_content='Institute of Engineering, Tribhuvan University, Nepal\nProblem Statement\n•   Eve

In [12]:
# Convert text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 139 texts..


Batches: 100%|██████████| 5/5 [00:02<00:00,  2.11it/s]

Generated embeddings with shape: (139, 384)
Adding 139 documents to vector store...
Successfully added 139 documents to vector store
Total documents in collection: 278


#### Retriever Pipleline from VectorStore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager= embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try: 
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

                
            
        

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("What is EventX")

Retrieving documents for query: 'What is EventX'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 77.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ce7b0c5c_2',
  'content': 'Institute of Engineering, Tribhuvan University, Nepal\nIntroduction\n• EventX is a centralized platform to view upcoming college events.\n• Displays events in a modern interactive calendar format.\n• Improves visibility and accessibility of event information.\n3',
  'metadata': {'doc_index': 2,
   'title': 'EventX-ppt-proposal',
   'content_length': 257,
   'file_type': 'pdf',
   'page': 2,
   'creationdate': '',
   'total_pages': 26,
   'page_label': '3',
   'creator': 'Google',
   'producer': 'PyPDF',
   'source': '../data/pdf/EventX-finalppt.pdf',
   'source_file': 'EventX-finalppt.pdf'},
  'similarity_score': 0.35764938592910767,
  'distance': 0.6423506140708923,
  'rank': 1},
 {'id': 'doc_513e8298_2',
  'content': 'Institute of Engineering, Tribhuvan University, Nepal\nIntroduction\n• EventX is a centralized platform to view upcoming college events.\n• Displays events in a modern interactive calendar format.\n• Improves visibility and access

### Integration VectorDB context pipeline with LLM output

In [16]:
import os
from dotenv import load_dotenv

load_dotenv()

os.getenv("OPENROUTER_API_KEY")

'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'

In [17]:
### Simple RAG pipeline with OpenRouter
import os
import requests

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is required. Set it in your environment.")

def openrouter_chat(prompt: str, model: str = "openai/gpt-oss-120b:free", max_tokens: int = 1024) -> str:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": max_tokens,
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using OpenRouter
    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""
    return openrouter_chat(prompt, max_tokens=1024)

answer = rag_simple("What is EventX?", rag_retriever)
print(answer)


Retrieving documents for query: 'What is EventX?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 132.46it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
EventX is a calendar‑based college event management system that lets students view, search and filter class and club events, see event details, receive notifications, and access timetables, while faculty, club admins and super admins can create, update, delete, and manage events, timetables, users and permissions.
